In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'libpysal', 'spreg', 'esda'], 
               capture_output=True, text=True)

# Verificar
import importlib
import libpysal
print(libpysal.__version__)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
PATH_PROCESSED   = "../data/Processed/"
PATH_MUNICIPIOS  = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
PATH_GEOJSON     = "../data/Processed/full_MIG.geojson"
PATH_NDVI        = "../data/landsat/landsat_ndvi/"
PATH_DEM         = "../data/dem/dem_srtm/"
PATH_GRUPOS_ARMADOS = "../data/simulaciones/grupos_armados_simulados.csv"

CRS              = "EPSG:4326"

# ============================================================
# CARGAR MUNICIPIOS
# ============================================================
municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs(CRS)

# ============================================================
# CARGAR X, y, grupos, coords desde archivos procesados
# ============================================================
X_list, y_list, grupos_list, coords_list = [], [], [], []

nombres_covariables = [
    'ndvi', 'altitud', 'pendiente',
    'dist_hidrografia', 'dist_vias',
    'dist_centros_poblados', 'dist_grupos_armados'
]

def extraer_valor_raster(raster_path, lon, lat):
    try:
        with rasterio.open(raster_path) as src:
            val = list(src.sample([(lon, lat)]))[0][0]
            if src.nodata and val == src.nodata:
                return np.nan
            return float(val)
    except:
        return np.nan

def calcular_pendiente_punto(dem_path, lon, lat, delta=0.001):
    try:
        with rasterio.open(dem_path) as src:
            z   = list(src.sample([(lon, lat)]))[0][0]
            z_e = list(src.sample([(lon+delta, lat)]))[0][0]
            z_n = list(src.sample([(lon, lat+delta)]))[0][0]
            dz_dx = (z_e - z) / (delta * 111320)
            dz_dy = (z_n - z) / (delta * 111320)
            return float(np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))))
    except:
        return np.nan

for pres_path in sorted(Path(PATH_PROCESSED).glob("presencias_*.csv")):
    codigo = pres_path.stem.replace("presencias_", "")
    cov_path = Path(PATH_PROCESSED) / f"covariables_{codigo}.npy"
    
    if not cov_path.exists():
        continue
    
    pres = pd.read_csv(pres_path)
    if len(pres) == 0:
        continue

    stack = np.load(cov_path)
    h_s, w_s = stack.shape[1], stack.shape[2]
    
    mun = municipios[municipios['MpCodigo'] == codigo]
    if len(mun) == 0:
        continue
    bounds = mun.total_bounds

    def c2p(lon, lat):
        col = np.clip(int((lon-bounds[0])/(bounds[2]-bounds[0])*w_s), 0, w_s-1)
        row = np.clip(int((bounds[3]-lat)/(bounds[3]-bounds[1])*h_s), 0, h_s-1)
        return row, col

    ndvi_path = Path(PATH_NDVI) / f"ndvi_{codigo}.tif"
    dem_path  = Path(PATH_DEM)  / f"dem_{codigo}.tif"

    # Presencias
    ause_path = Path(PATH_PROCESSED) / f"ausencias_{codigo}.csv"
    dfs = [pres]
    if ause_path.exists():
        ause = pd.read_csv(ause_path)
        if len(ause) > 0:
            dfs.append(ause)
    
    puntos = pd.concat(dfs, ignore_index=True)

    for _, punto in puntos.iterrows():
        lon, lat = punto['lon'], punto['lat']

        ndvi      = extraer_valor_raster(ndvi_path, lon, lat)
        altitud   = extraer_valor_raster(dem_path, lon, lat)
        pendiente = calcular_pendiente_punto(dem_path, lon, lat)

        r, c = c2p(lon, lat)
        dist_hidro = stack[3, r, c]
        dist_vias  = stack[4, r, c]
        dist_pob   = stack[5, r, c]
        dist_grupos   = stack[6, r, c]
        covs = np.array([ndvi, altitud, pendiente,
                         dist_hidro, dist_vias, dist_pob, dist_grupos])
        covs = np.where(np.isnan(covs), 0.0, covs)

        X_list.append(covs)
        y_list.append(float(punto['label']))
        grupos_list.append(codigo)
        coords_list.append((lon, lat))

X      = np.array(X_list)
y      = np.array(y_list)
grupos = np.array(grupos_list)
coords_array = np.array(coords_list)

print(f"✓ Total puntos: {len(X)}")
print(f"  Presencias: {int(y.sum())}")
print(f"  Ausencias: {int((y==0).sum())}")
print(f"  Municipios: {len(np.unique(grupos))}")
print(f"  NaN en X: {np.isnan(X).sum()}")

In [ ]:
import libpysal
import spreg
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ============================================================
# 1. CONSTRUIR COORDENADAS Y MATRIZ DE PESOS
# ============================================================

# Necesitamos las coordenadas de cada punto
# Las tenemos en coords_list que construimos al cargar X
# Si no está en memoria, reconstruirla:
coords_array = np.array(coords_list)  # shape: (n_puntos, 2) — lon, lat
print(f"Coordenadas: {coords_array.shape}")

# Matriz de pesos — K vecinos más cercanos
# K=5 es un buen punto de partida
w = libpysal.weights.KNN(coords_array, k=5)
w.transform = 'r'  # row-standardize
print(f"✓ Matriz de pesos construida: {w.n} puntos, {w.mean_neighbors:.1f} vecinos promedio")

# Verificar que la matriz es válida
print(f"  Islas (puntos sin vecinos): {w.islands}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import spreg
import libpysal
import numpy as np

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(X)),
    test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} puntos ({y_train.sum():.0f} presencias)")
print(f"Test:  {len(X_test)} puntos ({y_test.sum():.0f} presencias)")

# Normalizar
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Matriz de pesos solo para train
coords_train = coords_array[idx_train]
w_train = libpysal.weights.KNN(coords_train, k=5)
w_train.transform = 'r'
print(f"✓ Matriz de pesos: {w_train.n} puntos")

# GLM Espacial — Spatial Lag Model
modelo_glm = spreg.GM_Lag(
    y_train.reshape(-1, 1),
    X_train_sc,
    w=w_train,
    name_y='presencia',
    name_x=nombres_covariables
)

print(f"✓ Modelo entrenado")
print(f"\nCoeficientes:")
for nombre, beta in zip(['intercepto'] + nombres_covariables, 
                         modelo_glm.betas.flatten()):
    print(f"  {nombre:<25}: {beta:.4f}")

#print(f"  rho (lag espacial)       : {modelo_glm.rho:.4f}")

# Predicción en test
y_pred_linear = X_test_sc @ modelo_glm.betas[1:8] + modelo_glm.betas[0]
y_pred = 1 / (1 + np.exp(-y_pred_linear.flatten()))

auc = roc_auc_score(y_test, y_pred)
k = max(1, int(len(y_test) * 0.3))
prec_k = y_test[np.argsort(y_pred)[::-1][:k]].mean()

print(f"\nResultados 90-10:")
print(f"  AUC:          {auc:.3f}")
print(f"  Precision@k:  {prec_k:.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss

# ============================================================
# MÉTRICAS DE ORDENAMIENTO
# ============================================================
# Se usan probabilidades (predict_proba), no etiquetas 0/1:
# con etiquetas duras el ordenamiento queda lleno de empates y
# P@k depende del orden de las filas, no del modelo.
def metricas(y_true, y_score, frac_k=0.3):
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    k = max(1, int(len(y_true) * frac_k))
    top_k = np.argsort(-y_score, kind='stable')[:k]
    prevalencia = y_true.mean()
    prec_k = y_true[top_k].mean()
    return {
        'auc': roc_auc_score(y_true, y_score) if 0 < prevalencia < 1 else np.nan,
        'precision_at_k': prec_k,
        'recall_at_k': y_true[top_k].sum() / y_true.sum() if y_true.sum() > 0 else np.nan,
        'lift_at_k': prec_k / prevalencia if prevalencia > 0 else np.nan,
        'brier': brier_score_loss(y_true, y_score),
        'prevalencia': prevalencia,
    }

def nuevo_glm():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))

modelo_glm = LogisticRegression(max_iter=1000, random_state=42)
modelo_glm.fit(X_train_sc, y_train)

y_pred = modelo_glm.predict_proba(X_test_sc)[:, 1]
m_8020 = metricas(y_test, y_pred)

print(f"\nResultados 80-20:")
for nombre, valor in m_8020.items():
    print(f"  {nombre:<15}: {valor:.3f}")


In [ ]:
# Reservar municipios para evaluación final
municipios_holdout = ['17662', '70713', '05543']  
# Samaná, San Onofre, Peñol — tienen suficientes presencias

# Separar
idx_train = ~np.isin(grupos, municipios_holdout)
idx_test  =  np.isin(grupos, municipios_holdout)

X_train_f, X_test_f = X[idx_train], X[idx_test]
y_train_f, y_test_f = y[idx_train], y[idx_test]

# Entrenar modelo final (estandarización dentro del pipeline)
modelo_glm = nuevo_glm()
modelo_glm.fit(X_train_f, y_train_f)

# Evaluar
y_pred_f = modelo_glm.predict_proba(X_test_f)[:, 1]
m_holdout = metricas(y_test_f, y_pred_f)

print(f"Resultados reserva de municipios:")
for nombre, valor in m_holdout.items():
    print(f"  {nombre:<15}: {valor:.3f}")


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from tqdm import tqdm


print("\nValidación cruzada por municipio...")

logo = LeaveOneGroupOut()
resultados_cv = []
pred_oof = np.full(len(y), np.nan)  # predicciones fuera de municipio

splits = list(logo.split(X, y, grupos))

for train_idx, test_idx in tqdm(splits, desc="Validación cruzada"):
    municipio_test = grupos[test_idx][0]
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    modelo_glm = nuevo_glm()
    modelo_glm.fit(X_train, y_train)
    y_pred = modelo_glm.predict_proba(X_test)[:, 1]
    pred_oof[test_idx] = y_pred

    # AUC solo es definido si el municipio tiene presencias y ausencias
    if y_test.min() == y_test.max():
        continue

    m = metricas(y_test, y_pred)
    resultados_cv.append({'municipio': municipio_test,
                          'n_test': len(y_test),
                          'n_presencias_test': int(y_test.sum()),
                          **{c: round(v, 3) for c, v in m.items()}})

df_cv = pd.DataFrame(resultados_cv)
print(f"\nResultados CV:")
print(df_cv.to_string(index=False))
print(f"\nPromedio entre municipios (con presencias y ausencias):")
print(df_cv[['auc', 'precision_at_k', 'recall_at_k', 'lift_at_k', 'brier']].mean().round(3).to_string())

# Métricas agrupadas: todas las predicciones fuera de municipio juntas
m_oof = metricas(y, pred_oof)
print(f"\nAgrupado (todas las predicciones fuera de municipio):")
for nombre, valor in m_oof.items():
    print(f"  {nombre:<15}: {valor:.3f}")

df_cv.to_csv("../data/Processed/glm_resultados_cv.csv", index=False)
